# Multilingual Document OCR + MRZ Parser
## Domino Data Lab — Interactive Notebook

---

### Overview
This notebook performs **OCR on ID cards, passports, residency and bank documents** that may
contain **Arabic, French, and English** text.  
When a **Machine Readable Zone (MRZ)** is detected the notebook additionally parses and
validates it against the ICAO 9303 standard using **OmniMRZ**.

> ⚠️ **Fix applied — `AssertionError: param lang must in dict_keys(…), but got ar,fr,en`**  
> PaddleOCR only accepts **one language code per instance** — comma-separated values are not
> supported.  
> This notebook uses **two engines**: `lang='ar'` for Arabic script and `lang='fr'` for
> French / English Latin script. Results from both engines are merged and deduplicated.

### Key technology
| Component | Library | Notes |
|-----------|---------|-------|
| OCR engine ×2 | `paddleocr 2.8.1` | PP-OCRv4 mobile, CPU-only, one instance per script |
| ONNX inference | `onnxruntime 1.19.0` | `enable_hpi=True` — no GPU required |
| MRZ parsing | `omnimrz 0.1.6` | ICAO 9303 checksum validation |
| Excel output | `openpyxl 3.1.5` | Multi-sheet formatted workbook |

### Prerequisites
1. Upload `requirements.txt` to your Domino project.
2. In **Domino → Environments → Dockerfile instructions** add:
```dockerfile
RUN apt-get update && apt-get install -y --no-install-recommends libgl1 libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*
COPY requirements.txt /tmp/req.txt
RUN pip install --no-cache-dir -r /tmp/req.txt
```
3. Place document images under `/mnt/data/documents/`.
4. Run cells **top-to-bottom**.

### Expected outputs
| File | Description |
|------|-------------|
| `/mnt/artifacts/results.json` | Raw + structured OCR results, one object per document |
| `/mnt/artifacts/report.xlsx` | Three-sheet Excel report (Summary / MRZ Details / Full Text) |

---
## 1. Environment & Imports

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import json
import logging
import os
import re
import time
from pathlib import Path
from typing import Any

# ── Third-party ─────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

print('✓ Imports OK')

---
## 2. Configuration

Edit this cell to match your Domino workspace paths and preferences.

> **Language note:** PaddleOCR accepts **only one language code per instance**.  
> We define `LANG_ARABIC = 'ar'` for Arabic script and `LANG_LATIN = 'fr'` for French/English.

In [ ]:
# ── Domino workspace paths ───────────────────────────────────────────────────
INPUT_DIR    = Path('/mnt/data/documents')       # directory or single image
OUTPUT_JSON  = Path('/mnt/artifacts/results.json')
OUTPUT_EXCEL = Path('/mnt/artifacts/report.xlsx')

# ── OCR language codes — ONE per PaddleOCR engine ───────────────────────────
# FIX: passing 'ar,fr,en' raises AssertionError. Use separate engine instances.
LANG_ARABIC = 'ar'   # Arabic-script recognition model
LANG_LATIN  = 'fr'   # Latin-script model — covers French AND English

# ── Performance ──────────────────────────────────────────────────────────────
MAX_DIMENSION = 1280          # px — downscale images larger than this
CPU_THREADS   = os.cpu_count() or 4

# ── Supported image extensions ───────────────────────────────────────────────
SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}

# ── MRZ heuristic pattern ────────────────────────────────────────────────────
# ICAO MRZ lines contain only A-Z, 0-9 and '<' filler, minimum length 30
MRZ_LINE_PATTERN = re.compile(r'^[A-Z0-9<]{30,}$')

print('✓ Configuration')
print(f'  Arabic engine  → lang="{LANG_ARABIC}"')
print(f'  Latin engine   → lang="{LANG_LATIN}" (French + English)')
print(f'  CPU threads    → {CPU_THREADS}')
print(f'  Max image dim  → {MAX_DIMENSION} px')
print(f'  Input path     → {INPUT_DIR}')
print(f'  JSON output    → {OUTPUT_JSON}')
print(f'  Excel output   → {OUTPUT_EXCEL}')

---
## 3. Image Preprocessing

Images are resized to `MAX_DIMENSION` on their longest side before inference.  
`cv2.INTER_AREA` is used because it produces the best quality when downscaling.

In [ ]:
def load_and_resize(image_path: Path, max_dim: int = MAX_DIMENSION) -> np.ndarray:
    """
    Load an image and downscale it if the longest side exceeds *max_dim*.

    Parameters
    ----------
    image_path : Path  Path to the image file.
    max_dim    : int   Maximum allowed side length in pixels (default 1280).

    Returns
    -------
    np.ndarray  BGR image array ready for PaddleOCR.

    Raises
    ------
    FileNotFoundError  If OpenCV cannot open the file.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f'Cannot read image: {image_path}')

    h, w  = img.shape[:2]
    scale = min(max_dim / max(h, w), 1.0)   # never upscale
    if scale < 1.0:
        new_w = int(w * scale)
        new_h = int(h * scale)
        img   = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        log.debug('Resized %s  %dx%d → %dx%d', image_path.name, w, h, new_w, new_h)

    return img


print('✓ load_and_resize defined')

---
## 4. Initialise the Dual OCR Engines

### Why two engines?

PaddleOCR's `lang` parameter is validated against a fixed dictionary of supported codes:
```
dict_keys(['ch', 'en', 'korean', 'japan', 'chinese_cht', 'ta', 'te', 'ka', 'latin',
           'arabic', 'cyrillic', 'devanagari', 'ar', 'fr', ...])
```
Passing a comma-separated string like `'ar,fr,en'` is **not a valid key** and raises:
```
AssertionError: param lang must in dict_keys([…]), but got ar,fr,en
```
**Solution:** create one `PaddleOCR` instance per script family and merge their outputs.

### Engine settings
| Parameter | Value | Reason |
|-----------|-------|--------|
| `enable_hpi` | `True` | ONNX Runtime HPI — faster CPU inference |
| `use_angle_cls` | `False` | Documents assumed upright — saves time |
| `ocr_version` | `'PP-OCRv4'` | Lightest mobile model family |
| `use_gpu` | `False` | CPU-only (no GPU dependency) |

> **First run** downloads PP-OCRv4 model weights (~30 MB each). Subsequent runs use the cache.

In [ ]:
from paddleocr import PaddleOCR


def _make_engine(lang: str) -> PaddleOCR:
    """
    Create a single-language PaddleOCR engine.

    Parameters
    ----------
    lang : str  A single PaddleOCR language code, e.g. 'ar' or 'fr'.

    Returns
    -------
    PaddleOCR
    """
    return PaddleOCR(
        lang=lang,
        use_angle_cls=False,     # documents assumed upright
        use_gpu=False,           # CPU-only
        enable_hpi=True,         # ONNX Runtime HPI
        cpu_threads=CPU_THREADS,
        ocr_version='PP-OCRv4',  # lightest (mobile) model
        show_log=False,
    )


# ── Arabic engine ─────────────────────────────────────────────────────────────
print(f'Initialising Arabic engine  (lang="{LANG_ARABIC}") …')
ocr_ar = _make_engine(LANG_ARABIC)
print('  ✓ Arabic engine ready')

# ── Latin engine (French + English) ──────────────────────────────────────────
print(f'Initialising Latin engine   (lang="{LANG_LATIN}") …')
ocr_latin = _make_engine(LANG_LATIN)
print('  ✓ Latin engine ready  (French + English)')

log.info(
    'Both OCR engines ready  arabic=%s  latin=%s  threads=%d  HPI=True',
    LANG_ARABIC, LANG_LATIN, CPU_THREADS,
)

---
## 5. MRZ Detection & OmniMRZ Parsing

MRZ detection is a **two-stage process**:
1. **Heuristic scan** — regex checks each OCR text line for the ICAO MRZ character set.
2. **Structural parsing** — if ≥ 2 MRZ lines are found, OmniMRZ performs full ICAO 9303
   parsing and checksum validation.

In [ ]:
def detect_mrz_lines(text_lines: list[str]) -> list[str]:
    """
    Heuristic scan for MRZ-like lines in a list of OCR text strings.

    A line is considered an MRZ candidate if, after stripping whitespace
    and spaces (OCR sometimes inserts them), it matches ``^[A-Z0-9<]{30,}$``.
    Valid ICAO MRZ lines are exactly 30 (TD1), 36 (TD2) or 44 (TD3) chars;
    using ≥ 30 tolerates minor OCR artefacts.

    Parameters
    ----------
    text_lines : list[str]  Raw OCR lines from PaddleOCR.

    Returns
    -------
    list[str]  Cleaned MRZ line strings (internal spaces removed).
    """
    mrz_lines: list[str] = []
    for line in text_lines:
        cleaned = line.strip().replace(' ', '')
        if MRZ_LINE_PATTERN.match(cleaned):
            mrz_lines.append(cleaned)
    return mrz_lines


def parse_mrz_with_omnimrz(mrz_lines: list[str]) -> dict[str, Any]:
    """
    Call OmniMRZ to parse and validate a list of raw MRZ line strings.

    Parameters
    ----------
    mrz_lines : list[str]  MRZ lines from :func:`detect_mrz_lines`.

    Returns
    -------
    dict with keys:
        raw_mrz       — original line list
        parsed_fields — structured ICAO 9303 fields
        validation    — per-field checksums + overall_valid
        error         — present only if parsing raised an exception
    """
    try:
        from omnimrz import MRZ
    except ImportError:
        log.warning('omnimrz not installed — structural MRZ parsing skipped.')
        return {'error': 'omnimrz not installed', 'raw_mrz': mrz_lines}

    mrz_string = '\n'.join(mrz_lines)
    try:
        mrz_obj = MRZ(mrz_string)

        # ── Structured ICAO fields ───────────────────────────────────────
        fields: dict[str, Any] = {
            'document_type':    getattr(mrz_obj, 'document_type',    None),
            'issuing_country':  getattr(mrz_obj, 'issuing_country',  None),
            'document_number':  getattr(mrz_obj, 'document_number',  None),
            'surname':          getattr(mrz_obj, 'surname',          None),
            'given_names':      getattr(mrz_obj, 'given_names',      None),
            'nationality':      getattr(mrz_obj, 'nationality',      None),
            'birth_date':       getattr(mrz_obj, 'birth_date',       None),
            'sex':              getattr(mrz_obj, 'sex',              None),
            'expiry_date':      getattr(mrz_obj, 'expiry_date',      None),
            'optional_data_1':  getattr(mrz_obj, 'optional_data_1', None),
            'optional_data_2':  getattr(mrz_obj, 'optional_data_2', None),
        }

        # ── Validation / checksum flags ──────────────────────────────────
        validation: dict[str, Any] = {}
        for attr in dir(mrz_obj):
            if 'valid' in attr.lower() or 'check' in attr.lower():
                try:
                    validation[attr] = getattr(mrz_obj, attr)
                except Exception:
                    pass

        # Overall validity
        overall = getattr(mrz_obj, 'valid', None)
        if overall is None:
            overall = all(v for v in validation.values() if isinstance(v, bool))
        validation['overall_valid'] = overall

        return {
            'raw_mrz':       mrz_lines,
            'parsed_fields': fields,
            'validation':    validation,
        }

    except Exception as exc:
        log.warning('OmniMRZ parsing failed: %s', exc)
        return {'raw_mrz': mrz_lines, 'error': str(exc)}


print('✓ detect_mrz_lines defined')
print('✓ parse_mrz_with_omnimrz defined')

---
## 6. Single-Document Processing

Both engines run on the **same pre-processed image array**.  
Text lines are merged **Arabic first → Latin second**, with exact duplicates removed to avoid
double-counting lines that both engines recognised.

In [ ]:
def _extract_lines(engine: Any, img: np.ndarray) -> list[str]:
    """
    Run one PaddleOCR engine on *img* and return a list of text-line strings.

    PaddleOCR result structure:
        [ [ [box, (text, confidence)], [box, (text, confidence)], … ] ]

    Parameters
    ----------
    engine : PaddleOCR   An initialised PaddleOCR instance.
    img    : np.ndarray  BGR image array.

    Returns
    -------
    list[str]  Recognised text lines.
    """
    lines: list[str] = []
    raw = engine.ocr(img, cls=False)
    if raw and raw[0]:
        for item in raw[0]:
            if item and len(item) >= 2:
                txt = (
                    item[1][0]
                    if isinstance(item[1], (list, tuple))
                    else str(item[1])
                )
                lines.append(txt)
    return lines


def process_document(
    image_path:   Path,
    engine_ar:    Any,
    engine_latin: Any,
    max_dim:      int = MAX_DIMENSION,
) -> dict[str, Any]:
    """
    Run OCR on one image with both engines and optionally parse MRZ.

    Parameters
    ----------
    image_path   : Path        Path to the document image.
    engine_ar    : PaddleOCR   Arabic-script engine.
    engine_latin : PaddleOCR   Latin-script engine (French / English).
    max_dim      : int         Resize threshold in pixels.

    Returns
    -------
    dict with keys:
        file_name   — image filename
        full_text   — merged, deduplicated OCR lines (newline-joined)
        has_mrz     — True if ≥ 2 MRZ lines were detected
        mrz_data    — parsed MRZ dict or None
        error       — exception message or None
        elapsed_sec — wall-clock processing time in seconds
    """
    result: dict[str, Any] = {
        'file_name':   image_path.name,
        'full_text':   '',
        'has_mrz':     False,
        'mrz_data':    None,
        'error':       None,
        'elapsed_sec': 0.0,
    }
    t0 = time.perf_counter()

    try:
        img = load_and_resize(image_path, max_dim)

        # ── Run both engines on the same image ───────────────────────────
        lines_ar    = _extract_lines(engine_ar,    img)
        lines_latin = _extract_lines(engine_latin, img)

        # ── Merge: Arabic first then Latin; drop exact duplicates ────────
        seen: set[str]   = set()
        text_lines: list[str] = []
        for line in lines_ar + lines_latin:
            if line not in seen:
                seen.add(line)
                text_lines.append(line)

        result['full_text'] = '\n'.join(text_lines)
        log.debug(
            '%s  ar=%d  latin=%d  merged=%d',
            image_path.name, len(lines_ar), len(lines_latin), len(text_lines),
        )

        # ── MRZ detection & parsing ──────────────────────────────────────
        mrz_lines = detect_mrz_lines(text_lines)
        if len(mrz_lines) >= 2:
            result['has_mrz']  = True
            result['mrz_data'] = parse_mrz_with_omnimrz(mrz_lines)
            log.info('  ✓ MRZ detected (%d lines)', len(mrz_lines))
        else:
            log.info('  – No MRZ found')

    except Exception as exc:
        log.error('Error processing %s: %s', image_path.name, exc)
        result['error'] = str(exc)

    result['elapsed_sec'] = round(time.perf_counter() - t0, 3)
    return result


print('✓ _extract_lines defined')
print('✓ process_document defined  (dual-engine: ocr_ar + ocr_latin)')

---
## 7. Process a Single Sample Image

Change `SAMPLE_IMAGE` to point to a real document image in your workspace.  
Skip to **Section 8** to run the full batch.

In [ ]:
SAMPLE_IMAGE = Path('/mnt/data/documents/sample_passport.jpg')  # ← edit me

if SAMPLE_IMAGE.exists():
    print(f'Processing: {SAMPLE_IMAGE.name} …')
    single_result = process_document(SAMPLE_IMAGE, ocr_ar, ocr_latin)
    print(f'Done in {single_result["elapsed_sec"]:.2f}s  |  has_mrz={single_result["has_mrz"]}')
    print()
    print(json.dumps(single_result, ensure_ascii=False, indent=2, default=str))
else:
    print(f'[SKIP] {SAMPLE_IMAGE} not found — place an image there and re-run,')
    print('       or proceed directly to Section 8 for batch processing.')

---
## 8. Batch Processing

Processes every supported image under `INPUT_DIR` (configured in Section 2).  
Accepts both a single file path and a directory (searched recursively).

In [ ]:
def collect_images(input_path: Path) -> list[Path]:
    """
    Return a sorted list of image paths from a file or directory.

    Parameters
    ----------
    input_path : Path  Single image file or directory (searched recursively).

    Returns
    -------
    list[Path]
    """
    if input_path.is_file():
        return [input_path]
    images = sorted(
        p for p in input_path.rglob('*')
        if p.suffix.lower() in SUPPORTED_EXT
    )
    if not images:
        log.warning('No supported images found in %s', input_path)
    return images


# ── Discover images ───────────────────────────────────────────────────────────
images = collect_images(INPUT_DIR)
log.info('Found %d image(s) under %s', len(images), INPUT_DIR)

# ── Run batch ─────────────────────────────────────────────────────────────────
all_results: list[dict[str, Any]] = []

for img_path in tqdm(images, desc='OCR + MRZ', unit='doc'):
    log.info('Processing: %s', img_path.name)
    r = process_document(img_path, ocr_ar, ocr_latin)
    all_results.append(r)
    log.info('  Done in %.2fs  |  has_mrz=%s  |  error=%s',
             r['elapsed_sec'], r['has_mrz'], r['error'])

total_time = sum(r['elapsed_sec'] for r in all_results)
print(f'\n✓ Batch complete — {len(all_results)} document(s) processed in {total_time:.1f}s')

---
## 9. Save Results as JSON

Results are written as a pretty-printed, UTF-8 JSON array (one object per document).  
Parent directories are created automatically if they do not exist.

In [ ]:
def save_json(results: list[dict[str, Any]], output_path: Path) -> None:
    """
    Write *results* to a pretty-printed UTF-8 JSON file.

    Parameters
    ----------
    results     : list[dict]  OCR result dictionaries.
    output_path : Path        Destination file path.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2, default=str)
    size_kb = output_path.stat().st_size / 1024
    log.info('JSON saved → %s  (%.1f KB)', output_path, size_kb)


save_json(all_results, OUTPUT_JSON)
print(f'✓ JSON saved → {OUTPUT_JSON}')

---
## 10. Convert JSON to Formatted Excel

Creates a three-sheet workbook with bold headers and auto-fitted column widths.

| Sheet | Content |
|-------|---------|
| **Summary** | One row per document — filename, has_mrz, overall MRZ validity, timing, errors |
| **MRZ Details** | Flattened ICAO 9303 fields (documents with MRZ only) |
| **Full Text** | Raw merged OCR text for every document |

In [ ]:
def _apply_header_style(ws) -> None:
    """Bold font + light-blue fill on the header row."""
    fill = PatternFill(fill_type='solid', fgColor='BDD7EE')
    for cell in ws[1]:
        cell.font      = Font(bold=True)
        cell.fill      = fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)


def _autofit_columns(ws, max_width: int = 60) -> None:
    """Set column widths based on max content length (approximate)."""
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        best   = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[letter].width = min(best + 4, max_width)


print('✓ Excel formatting helpers defined')

In [ ]:
def build_excel(results: list[dict[str, Any]], output_path: Path) -> None:
    """
    Build a formatted multi-sheet Excel report from *results*.

    Sheets: Summary | MRZ Details | Full Text

    Parameters
    ----------
    results     : list[dict]  OCR result dictionaries.
    output_path : Path        Destination .xlsx file path.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    summary_rows: list[dict] = []
    mrz_rows:     list[dict] = []
    text_rows:    list[dict] = []

    for r in results:
        mrz_data   = r.get('mrz_data') or {}
        validation = mrz_data.get('validation', {}) if r.get('has_mrz') else {}
        overall    = validation.get('overall_valid')

        # ── Summary row ───────────────────────────────────────────────────
        summary_rows.append({
            'file_name':         r.get('file_name'),
            'has_mrz':           r.get('has_mrz'),
            'overall_mrz_valid': overall,
            'elapsed_sec':       r.get('elapsed_sec'),
            'error':             r.get('error'),
        })

        # ── Full-text row ─────────────────────────────────────────────────
        text_rows.append({
            'file_name': r.get('file_name'),
            'full_text': r.get('full_text'),
        })

        # ── MRZ detail row (MRZ documents only) ───────────────────────────
        if r.get('has_mrz') and mrz_data:
            pf = mrz_data.get('parsed_fields', {}) or {}
            mrz_rows.append({
                'file_name':       r.get('file_name'),
                'raw_mrz':         ' | '.join(mrz_data.get('raw_mrz', [])),
                'document_type':   pf.get('document_type'),
                'issuing_country': pf.get('issuing_country'),
                'document_number': pf.get('document_number'),
                'surname':         pf.get('surname'),
                'given_names':     pf.get('given_names'),
                'nationality':     pf.get('nationality'),
                'birth_date':      pf.get('birth_date'),
                'sex':             pf.get('sex'),
                'expiry_date':     pf.get('expiry_date'),
                'optional_data_1': pf.get('optional_data_1'),
                'optional_data_2': pf.get('optional_data_2'),
                'overall_valid':   validation.get('overall_valid'),
                # Individual validation flags as columns
                **{
                    f'valid_{k}': v
                    for k, v in validation.items()
                    if k != 'overall_valid'
                },
            })

    # ── Build DataFrames ──────────────────────────────────────────────────────
    df_summary = pd.DataFrame(summary_rows)
    df_mrz     = (
        pd.DataFrame(mrz_rows)
        if mrz_rows
        else pd.DataFrame(columns=['file_name'])
    )
    df_text = pd.DataFrame(text_rows)

    # ── Write sheets ──────────────────────────────────────────────────────────
    with pd.ExcelWriter(str(output_path), engine='openpyxl') as writer:
        df_summary.to_excel(writer, sheet_name='Summary',     index=False)
        df_mrz.to_excel(    writer, sheet_name='MRZ Details', index=False)
        df_text.to_excel(   writer, sheet_name='Full Text',   index=False)

    # ── Apply formatting ──────────────────────────────────────────────────────
    wb = load_workbook(str(output_path))
    for ws in wb.worksheets:
        _apply_header_style(ws)
        _autofit_columns(ws)
    wb.save(str(output_path))

    size_kb = output_path.stat().st_size / 1024
    log.info('Excel saved → %s  (%.1f KB)', output_path, size_kb)


build_excel(all_results, OUTPUT_EXCEL)
print(f'✓ Excel report saved → {OUTPUT_EXCEL}')

---
## 11. Quick Summary Preview

In [ ]:
df_summary = pd.read_excel(str(OUTPUT_EXCEL), sheet_name='Summary')

print('=' * 42)
print('  BATCH SUMMARY')
print('=' * 42)
print(f'  Total documents   : {len(df_summary)}')
print(f'  With MRZ          : {int(df_summary.has_mrz.sum())}')
print(f'  MRZ valid         : {int(df_summary.overall_mrz_valid.sum())}')
print(f'  Processing errors : {int(df_summary.error.notna().sum())}')
print('=' * 42)

df_summary

---
## 12. Performance Summary

In [ ]:
times = [r['elapsed_sec'] for r in all_results if r.get('elapsed_sec')]

if times:
    print(f'Documents processed : {len(times)}')
    print(f'Total time          : {sum(times):.1f} s')
    print(f'Mean time / doc     : {sum(times) / len(times):.2f} s')
    print(f'Min  / Max          : {min(times):.2f} s / {max(times):.2f} s')
else:
    print('No timing data available.')

---
## Appendix A — Domino Environment Setup

### Option 1 – Dockerfile instructions (recommended)
Paste into **Domino → Environments → Dockerfile instructions**:
```dockerfile
# System libraries needed by OpenCV
RUN apt-get update && apt-get install -y --no-install-recommends \
    libgl1 libglib2.0-0 && rm -rf /var/lib/apt/lists/*

COPY requirements.txt /tmp/req.txt
RUN pip install --no-cache-dir -r /tmp/req.txt
```

### Option 2 – Pre-run script
In **Project Settings → Pre-run script**:
```bash
pip install \
  paddleocr==2.8.1 paddlepaddle==2.6.1 \
  onnxruntime==1.19.0 omnimrz==0.1.6 \
  opencv-python-headless==4.10.0.84 Pillow==10.4.0 \
  numpy==1.26.4 pandas==2.2.2 openpyxl==3.1.5 tqdm==4.66.5
```

### Workspace layout
```
/mnt/data/documents/    ←  place input images here
/mnt/artifacts/         ←  results.json and report.xlsx written here
/mnt/code/              ←  this notebook and multilingual_doc_ocr.py
```

---
## Appendix B — PaddleOCR Supported Language Codes

| Code | Script / Language |
|------|-------------------|
| `ar` | Arabic |
| `fr` | French (also recognises English Latin characters) |
| `en` | English only |
| `ch` | Simplified Chinese |
| `korean` | Korean |
| `japan` | Japanese |
| `devanagari` | Hindi / Devanagari script |

> ⚠️ **Each PaddleOCR instance accepts exactly one code from this list.**  
> Comma-separated values (e.g. `'ar,fr,en'`) raise an `AssertionError`.  
> Always use one engine per script, as demonstrated in Section 4.